In [ ]:
# Setup
import sys
import os

# Check if the code is running in Google Colab
if 'google.colab' in sys.modules:
    print("Detected Google Colab environment.")

    # 1. Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # 2. Define the path to your project folder in Drive
    # IMPORTANT: This is the name of my drive folder. Either make one with the same name, or update the string to your own folder.
    project_path = '/content/drive/My Drive/clinical_trials_embeddings/'

    # 3. Copy utils.py to the local Colab session
    # This makes sure that the utils.py file can be accesed. You can also drag and drop it into the local colab session
    # If your runtime times out, you will need to rerun this or rerun the drag and drop so that the code has access to the utils file
    utils_source = os.path.join(project_path, 'utils.py')

    if os.path.exists(utils_source):
        print(f"Found utils.py at {project_path}")
        print("Copying to local session...")
        !cp "{utils_source}" .
        print("utils.py loaded successfully.")
    else:
        print(f"ERROR: utils.py not found at {utils_source}")
        print("Please check the 'project_path' variable.")

    # 4. Install Dependencies from requirements.txt
    req_path = os.path.join(project_path, 'requirements.txt')
    if os.path.exists(req_path):
        print("Installing dependencies from requirements.txt...")
        !pip install -q -r "{req_path}"
        print("Dependencies installed.")
    else:
        print("requirements.txt not found. Falling back to manual install.")
        !pip install -q sentence-transformers datasets

else:
    print("Detected Local environment.")
    project_path = './'

#  Introduction
# # Project: Predicting Clinical Trial Success through Multimodal Data Fusion

This notebook presents the final execution and analysis of our project on predicting clinical trial outcomes. Working with the full **louisbrulenaudet/clinical-trials** dataset (~540,000 trials), we aim to show that combining structured metadata with semantic text embeddings allows for better risk detection compared to unimodal approaches.

A central challenge in this domain is the real-world class imbalance, where successful trials vastly outnumber failures. In this notebook, we implement a  pipeline to address this, moving from naive baselines to a balanced training strategy, and culminating in a custom Multimodal Fusion Neural Network.

The notebook is structured as follows:

1.  **Exploratory Data Analysis (EDA):** An examination of the full dataset's structured features and the extent of class imbalance.
2.  **Feature Engineering (Text Embeddings):** Documentation of the generation of semantic embeddings using `BioClinical-ModernBERT`.
3.  **Data Preparation:** Loading the full dataset and creating Train/Validation/Test splits. *Note: we split the data before balancing to make sure our Test set reflects real-world conditions.*
4.  **Baselines Phase 1 (Unbalanced):** Training Random Forest models on the raw data to establish a baseline and demonstrate the "accuracy paradox" (high accuracy, zero recall).
5.  **Data Balancing Strategy:** Implementing a random downsampling strategy on the training set to recover the predictive signal for failed trials.
6.  **Baselines Phase 2 (Balanced):** Retraining the structured and embedding models to quantify the improvement in Recall (failure detection).
7.  **Multimodal Fusion Model:** Implementing and training a PyTorch-based **Intermediate Fusion Neural Network** on a GPU, combining both data modalities.
8.  **Post-Hoc Analysis & Conclusion:** Optimizing decision thresholds to balance Precision/Recall and presenting a final comparative analysis of all five models.

In [ ]:
# Libraries

# Core Data Handling & System
import pandas as pd
import numpy as np
import os
import gc

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.utils import resample

# Utility Functions (from our utils.py file)
from utils import (
    fit_and_preprocess_train,
    transform_test_data,
    balance_raw_data,
    ClinicalTrialDataset,
    train_model
)

# Configuration
sns.set_style("whitegrid")
pd.options.display.float_format = '{:.2f}'.format
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load the original dataset (excludes embeddings)

# We load the 'train' split, which contains all the available data,
# and immediately convert it to a pandas DataFrame for EDA.

print("Loading the full dataset from Hugging Face...")
full_df = load_dataset("louisbrulenaudet/clinical-trials", split="train").to_pandas()
print("Dataset loaded.")

# Display initial information about the dataset
print("\nDataset Information:")
full_df.info()

print("\nFirst 5 rows of the raw dataset:")
display(full_df.head())

# Exploratory Data Analysis (EDA)

This section focuses on developing a grounded understanding of the clinical trials dataset. Our aim is to understand how the data are structured, examine key patterns and distributions, and identify potential issues such as missing values or class imbalance. These insights form the foundation for the preprocessing and modeling steps that follow. All analyses here use the full, unfiltered dataset.

### Structured Feature Analysis

We begin with the structured features that are most likely to influence trial outcomes. These variables were selected for their ability to capture aspects of study design, scale, and participant characteristics:

* **`phases`, `study_type`, `enrollment_count`:** Describe the trial’s stage, format, and size, which together indicate its overall complexity and risk.
* **`lead_sponsor_class`:** Represents the funding source (e.g., industry or academic sponsor), which may correlate with trial design and outcome likelihood.
* **`sex`, `minimum_age`, `maximum_age`:** Define the population under study, helping us gauge demographic scope and potential variability.

Our following analysis examines the extent of missing data and the distributions of these categorical and numerical variables. These findings will guide the development of preprocessing methods and inform early model design decisions.

In [ ]:
# Target Variable Analysis: overall_status

# First, we look at the original distribution of the 'overall_status' column.
print("--- Original Distribution of Trial Statuses ---")
status_counts = full_df['overall_status'].value_counts()
print(status_counts)

# Next we visualize the distribution
plt.figure(figsize=(12, 7))
sns.barplot(x=status_counts.index, y=status_counts.values, palette="viridis")
plt.title('Original Distribution of Overall Trial Status', fontsize=16)
plt.xlabel('Status', fontsize=12)
plt.ylabel('Number of Trials', fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Defining the Binary Target for EDA

The `overall_status` column contains many different categories, where a lot of them are not definitive outcomes (e.g., "RECRUTING"). For our predictive task, we need a clear binary target representing **Success vs. Failure**. Based on the above plot, we define our classes as follows:

*   **Success (Target = 1):** Trials marked **"COMPLETED"**.
*   **Failure (Target = 0):** Trials that were stopped for definitive negative reasons. We include **"TERMINATED"**, **"WITHDRAWN"**, and **"SUSPENDED"**.

We will filter the dataset to these outcomes to create a clean target for our analysis.

In [ ]:
# Create a temporary filtered DataFrame for EDA visualization

success_stati = ["COMPLETED"]
failure_stati = ["TERMINATED", "WITHDRAWN", "SUSPENDED"]
definitive_stati = success_stati + failure_stati

# Note: This filtered_df is for EDA purposes only.
# The final filtering for the model will happen within the preprocessing pipeline.
eda_filtered_df = full_df[full_df['overall_status'].isin(definitive_stati)].copy()
eda_filtered_df['target'] = eda_filtered_df['overall_status'].apply(lambda x: 1 if x in success_stati else 0)

# Visualize the new binary target distribution
print("--- Distribution of the Binary Target Variable ---")
print(eda_filtered_df['target'].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=eda_filtered_df, palette="viridis")
plt.title('Distribution of Binary Target (0=Failure, 1=Success)', fontsize=14)
plt.show()

# Markdown analysis of the imbalance will go in the final EDA conclusion.

In [ ]:
# Missing Values Analysis

structured_features = [
    'phases', 'study_type', 'enrollment_count',
    'lead_sponsor_class', 'sex', 'minimum_age', 'maximum_age'
]

missing_values = eda_filtered_df[structured_features].isnull().sum()
missing_percentage = (missing_values / len(eda_filtered_df)) * 100

print("--- Percentage of Missing Values in Key Features ---")
print(missing_percentage[missing_percentage > 0].sort_values(ascending=False))

**Interpretation:**
The missing value analysis reveals several different frequencies of missing values, each with their own set of solutions or challenges:
*   `sex` and `enrollment_count` have very few missing values (<2%), making it safe to simply drop these rows in our processing pipeline.
*   `phases`, `minimum_age`, and `maximum_age` have a moderate to high percentage of missing data (6% to 47%). For these, we will need to use an imputation strategy (e.g., filling with "Unknown" for categoricals or the median for numericals) to avoid losing a significant portion of our dataset. This is the info we needed to design our preprocessing functions.

In [ ]:
# Categorical Feature Distributions

# 1. 'phases' column (which contains lists/ndarrays)
print("--- Distribution of Trial Phases ---")
phase_counts = eda_filtered_df['phases'].explode().value_counts()
plt.figure(figsize=(10, 6))
sns.barplot(x=phase_counts.index, y=phase_counts.values, palette='viridis')
plt.title('Distribution of Trial Phases', fontsize=16)
plt.xticks(rotation=45, ha="right")
plt.show()

# 2. Other categorical features
simple_categorical_features = ['study_type', 'lead_sponsor_class', 'sex']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distributions of Other Categorical Features', fontsize=20)
axes = axes.flatten()
for i, feature in enumerate(simple_categorical_features):
    sns.countplot(x=feature, data=eda_filtered_df, ax=axes[i], palette='viridis', order=eda_filtered_df[feature].value_counts().index)
    axes[i].set_title(f'Distribution of {feature}')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

**Interpretation:**

* **Phases:** The distribution is largely dominated by “NA” (Not Applicable) entries. We interpret these as corresponding to observational studies, which typically do not include defined phases. This category is therefore meaningful in itself, rather than simply representing missing data.
* **Sponsor:** The `lead_sponsor_class` feature is very skewed toward “OTHER.” While this limits its immediate interpretability, it still helps distinguish between major sponsor types such as “INDUSTRY” and “NIH.” A more granular manual grouping of these could potentially yield better insights. However, we hypothesize that our Multimodal Fusion Model will captures some of this institutional nuance implicitly through the trial summaries (embeddings), reducing the need for explicit manual feature engineering.
* **Study Type & Sex:** These features are imbalanced, reflecting the reality of clinical research (more interventional studies, most open to all sexes). This is not a data problem, but rather a real-world pattern our model can learn from.


In [ ]:
# Numerical Feature Analysis

numerical_features = ['enrollment_count', 'minimum_age', 'maximum_age']

# 1. Check data types and summary statistics
print("--- Data Types of Numerical Features ---")
print(eda_filtered_df[numerical_features].dtypes)
print("\n--- Summary Statistics for Numerical Features ---")
display(eda_filtered_df[numerical_features].describe())

**Interpretation of Statistics:**
The summary statistics table is highly revealing:
*   **Skewness:** For all three features, the `mean` is significantly different from the `median` (50%), indicating strong skewness. `enrollment_count` shows the most extreme right skew.
*   **Outliers/Data Quality:** The age columns contain clear data errors or extreme outliers, with a `max` value of 730 for `minimum_age` and 6569 for `maximum_age`.  Building on our findings from P2, where we identified biologically impossible outliers in the age columns (e.g., 6000 years), we have implemented a capping strategy in our preprocessing pipeline. We clip all ages to the [0, 120] range. This is particularly critical for the P3 Neural Network model, which relies on Z-score standardization and would otherwise be distorted by these extreme values.

In [ ]:
# Plot distributions (Linear vs. Log Scale)
fig, axes = plt.subplots(len(numerical_features), 2, figsize=(12, 12))
fig.suptitle('Distributions of Numerical Features (Linear vs. Log Scale)', fontsize=16)
for i, feature in enumerate(numerical_features):
    sns.histplot(eda_filtered_df[feature], bins=50, kde=False, ax=axes[i, 0]).set_title(f'{feature} (Linear Scale)')
    sns.histplot(eda_filtered_df[feature], bins=50, kde=False, ax=axes[i, 1], log_scale=True).set_title(f'{feature} (Log Scale)')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# "Zoomed-in" plot for minimum_age
zoomed_df = eda_filtered_df[eda_filtered_df['minimum_age'] <= 100]
plt.figure(figsize=(12, 7))
sns.histplot(data=zoomed_df, x='minimum_age', bins=100)
plt.title('Zoomed-In Distribution of Minimum Age (0-100 years)', fontsize=16)
plt.xticks(ticks=range(0, 101, 5))
plt.show()

**Interpretation of Plots:**
*   The log-scaled plots confirm the right-skew of `enrollment_count` and `maximum_age`.
*   The zoomed-in plot for `minimum_age` tells us the most. It proves that this feature does not have a smooth distribution and instead acts like a combination of a smooth distribution at low ages and a **categorical variable**, dominated by specific values like 0, 18, and other "round number" age cutoffs (40, 50, 65). This is likely because many studies aimed at lower age groups have specific age requirements where a few years' difference is developmentally significant (e.g., 13 vs. 16 years old), while studies targeting older adults often use broader, round-number cutoffs where the difference between 40 and 42 is less critical. We considered binning this feature, but we chose to leave it as a continuous numerical variable (capped at 120) to allow the non-linear models (Random Forest and Neural Network) to learn the precise age-related boundaries without the information loss associated with manual binning.

## EDA Conclution

Our exploratory analysis has provided several good insights that directly inform our project's next steps:
1.  **A Clear (but Imbalanced) Target:** We have successfully defined a binary target variable ("Success" vs. "Failure"), but its severe class imbalance is the primary challenge we must address.
2.  **Informative (but Messy) Features:** The structured features show clear patterns and distributions that we **hypothesize contain predictive signals** (e.g., the difference in `phases` or `study_type`). However, they also suffer from missing data, skewness, and data quality issues (e.g., age outliers). Our preprocessing pipeline must be designed to handle these issues robustly to effectively test this hypothesis.

With this new and deeper understanding of the data, we are now ready to move on to the formal data preparation and modeling phase.

#  Feature Engineering: Text Embeddings

One of the hypothesis of this project is that the structured metadata alone (e.g., Phase, Enrollment Count) is not enough to capture the complexity of clinical trial failure. The "scientific logic" of a trial is often buried in its textual descriptions.

### Embeddings Generation Strategy
While the original dataset documentation stated that it included pre-computed embeddings, they were absent from the dataset files. So, we engineered these features manually to enable our Multimodal approach.

We utilized the **`thomas-sounack/BioClinical-ModernBERT-base`** model, a transformer architecture pre-trained specifically on biomedical and clinical text. This model was chosen for its ability to capture domain-specific semantic nuances that generic BERT models might miss, as well as being reccomended by the dataset creators. We generated embeddings for two key text columns:
1.  **`brief_summary`**: Captures the trial's design and intent.
2.  **`eligibility_criteria`**: Captures the inclusion/exclusion logic, often a key determinant of recruitment failure.

Each text field was converted into a **768-dimensional vector**, resulting in a combined **1536-dimensional feature space** for the text modality.

### Offline Computation and Sharding
Due to the computational expense of processing ~540,000 trials through a Transformer model, this step was performed offline using a GPU-accelerated environment. To manage memory constraints and prevent timeouts, we adopted a **sharding strategy**:
1.  **Sharding:** The raw dataset was split into manageable chunks (shards).
2.  **Processing:** Each shard was processed individually using the `embedding-generation.ipynb` notebook, found in the project repository.
3.  **Merging:** The processed shards were reassembled into the final dataset using a dedicated script, `combine.py`, found in the archive folder of the repository.

The resulting dataset, containing both the original structured data and the new high-dimensional embeddings, was saved as a Parquet file for efficient loading.

# Data Loading & Splitting

With the addition of the 1536-dimensional embeddings, the memory footprint of the full dataset (approx. 540,000 trials) exceeds the RAM limits of standard cloud environments when loaded as a single Pandas DataFrame (12 GB in the base Colab). Trying to load the file and perform a standard `train_test_split` in-memory results in immediate runtime crashes.

To fix this, we performed the data partitioning in a dedicated upstream notebook (`splits.ipynb`). We utilized **Polars**, a high-performance dataframe library, to efficiently shuffle and split the data while minimizing memory overhead.

The data was partitioned into three distinct sets using **stratified sampling** to maintain the original class imbalance across all splits:
1.  **Training Set (70%):** Used for model training and balancing.
2.  **Validation Set (15%):** Used for hyperparameter tuning and early stopping during neural network training.
3.  **Test Set (15%):** A strict hold-out set used exclusively for final evaluation.

By physically separating these subsets into individual Parquet files, we achieve two goals: effective memory management (loading only what is needed) and a guarantee of zero data leakage, as the test set remains isolated from all preprocessing and balancing steps.

In [ ]:
# Load the Full Parquet files (Train/Val/Test) from Drive.

# 1. Define the path where the files are located
# This should match the 'output_dir' from the Polars script we ran
base_path = '/content/drive/My Drive/clinical_trials_embeddings/'

print(f"Loading datasets from: {base_path}")

# 2. Load TRAIN set
# We add memory_map=True to help Colab manage RAM
X_train = pd.read_parquet(base_path + 'train_set.parquet', memory_map=True)
y_train = X_train['overall_status']

# 3. Load VALIDATION set
X_val = pd.read_parquet(base_path + 'val_set.parquet', memory_map=True)
y_val = X_val['overall_status']

# 4. Load TEST set
# Tip: If your Colab session crashes due to RAM here, comment out the next two lines
# and only load the test set at the very end of your notebook when you need it.
X_test = pd.read_parquet(base_path + 'test_set.parquet', memory_map=True)
y_test = X_test['overall_status']

print(f"Final Shapes:")
print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

In [ ]:
# RAM OPTIMIZATION: Separate Structured from Embeddings

# 1. Define the columns we need for structured preprocessing
# (These must match what's in your utils.py)
structured_cols = [
    'phases', 'study_type', 'enrollment_count',
    'lead_sponsor_class', 'sex', 'minimum_age', 'maximum_age'
]

# 2. Create lightweight dataframes for structured processing
# We keep the index to align them later!
print("Separating structured data...")
X_train_struct_raw = X_train[structured_cols].copy()
X_val_struct_raw = X_val[structured_cols].copy()
X_test_struct_raw = X_test[structured_cols].copy()

# 3. Create Target variables
# (You already did this, but ensuring they are separate series helps)
y_train = X_train['overall_status']
y_val = X_val['overall_status']
y_test = X_test['overall_status']

print("Structured data separated.")

## Split before balancing

An important methodological decision we made for this pipeline is the strict separation of **Data Splitting** and **Data Balancing**.

In the following sections, we will employ a downsampling strategy to address class imbalance. However, it is important that this balancing is applied **exclusively to the Training set**. The Validation and Test sets must remain in their original, imbalanced state for two reasons:

1.  **Reflecting Real-World Distributions:** The prevalence of trial failure in our test dataset is approximately 14%. If we were to balance the Test set (making it 50% failure), our evaluation metrics—particularly Precision and F1-Score—would be artificially inflated and would not reflect the model's performance in a real deployment scenario.
2.  **Preventing Data Leakage:** Modifying the distribution of the test data constitutes a form of data leakage, where the evaluation conditions are engineered to suit the model rather than the reality of the problem.

Therefore, throughout the rest of this notebook, you will observe that while the training data is downsampled to a 50/50 ratio to facilitate feature learning, all reported performance metrics are derived from the original, highly imbalanced Test set.

## Structured Data Preprocessing Pipeline

To make sure our preprocessing is modular, reusable, and consistently applied, we have created two functions, stored in a separate `utils.py` file. This `fit`/`transform` paradigm is a standard best practice that prevents data leakage from the test set into the training process.

### Utility Functions
1.  **`fit_and_preprocess_train(train_df)`:** This function learns transformation parameters (like medians and scaling factors) exclusively from the training data and applies them. It returns a processed training DataFrame and a `transformation_rules` dictionary.
2.  **`transform_test_data(test_df, rules)`:** This function takes the test data and the `transformation_rules` dictionary and applies the already learned transformations, ensuring the test set is treated as unseen data.

### Detailed Preprocessing Steps
The pipeline implemented in our utility functions performs the following sequential operations on the structured data:

*   **1. Feature Selection:** It selects the seven structured features identified during EDA: `phases`, `study_type`, `enrollment_count`, `lead_sponsor_class`, `sex`, `minimum_age`, and `maximum_age`.
*   **2. Outcome Filtering:** It filters the dataset to only include trials with a definitive outcome ("COMPLETED", "TERMINATED", "WITHDRAWN", "SUSPENDED").
*   **3. Handling Missing Values & Outliers:**
    *   **Row Dropping:** Drops rows where `sex` or `enrollment_count` are missing (<2% of data).
    *   **Categorical Imputation:** Fills missing `NaN` values in `phases` with `"Unknown"`.
    *   **Numerical Imputation:** Fills missing `NaN` values in age columns with the training set median.
    *   **Outlier Capping:** Clips `minimum_age` and `maximum_age` to the range [0, 120] to prevent extreme outliers (e.g., 6000 years) from distorting the standardization process.
*   **4. One-Hot Encoding:** Converts categorical features into a numerical format. The encoder uses `handle_unknown='ignore'` to robustly handle unseen categories in the test set.
*   **5. Numerical Scaling:** Standardizes numerical features using a `StandardScaler` (mean=0, std=1) to ensure efficient training for the Neural Network.

In [ ]:
# Preprocess Structured Data (Unbalanced)

# We need to temporarily attach the target to X_train_struct_raw
# because fit_and_preprocess_train expects it for filtering
X_train_struct_raw['overall_status'] = y_train

print("Preprocessing Unbalanced Train...")
processed_train_struct, rules = fit_and_preprocess_train(X_train_struct_raw)

# Now we prepare Val and Test for transformation
# We must attach 'overall_status' to them too because transform_test_data expects it
X_val_struct_raw['overall_status'] = y_val
X_test_struct_raw['overall_status'] = y_test

print("Preprocessing Val/Test...")
processed_val_struct = transform_test_data(X_val_struct_raw, rules)
processed_test_struct = transform_test_data(X_test_struct_raw, rules)

print("Structured preprocessing complete.")
print(f"Processed Train Shape: {processed_train_struct.shape}")

# Baselines Phase 1: The Impact of Imbalance

To quantify the value of our proposed interventions (Data Balancing and Multimodal Fusion), we must first establish a baseline. In this phase, we train our baseline, uncombined models on the **raw, unbalanced training set** (approximately 86% Success / 14% Failure) and evaluate them on the unbalanced test set.

Our primary goal here is to show the baseline behavior of standard machine learning algorithms when faced with severe class imbalance: the tendency to maximize global accuracy by ignoring the minority class (the "Lazy Classifier" phenomenon).

### Model Selection Rationale: Random Forest
For both the Structured Data and the Text Embeddings, we utilize a **Random Forest Classifier** as our baseline. This choice is driven by specific requirements for each data modality:

*   **For Structured Metadata:** Clinical trial data involves complex, non-linear interactions between features (e.g., the risk associated with a specific *Sponsor* type might depend heavily on the *Phase* of the trial). Random Forests are the industry standard for tabular data, offering robustness to outliers and the ability to capture these non-linear decision boundaries without extensive hyperparameter tuning.

*   **For Text Embeddings:** In our preliminary P2 analysis, we attempted to classify trial outcomes using linear models (Logistic Regression) on the embeddings. These models failed to achieve meaningful predictive power (F1-scores < 0.10), suggesting that the relationship between the semantic embedding space and trial success is highly non-linear. Consequently, we require a baseline with sufficient capacity to model complex manifolds in the 1536-dimensional feature space.

In the following cells, we train these baselines and analyze their Confusion Matrices to diagnose the specific failure modes caused by the class imbalance.

In [ ]:
# Run Baseline 1 (Structured Unbalanced).

# Use the specific variables we created in "Preprocess Structured Data (Unbalanced)"
# processed_train_struct contains the target column 'target'
X_train_s = processed_train_struct.drop('target', axis=1)
y_train_s = processed_train_struct['target']

# We use the TEST set here for the final report numbers
X_test_s = processed_test_struct.drop('target', axis=1)
y_test_s = processed_test_struct['target']

# --- Sanity Check ---
print("--- Final Shapes for Structured Model (Unbalanced) ---")
print(f"Shape of X_train: {X_train_s.shape}")
print(f"Shape of y_train: {y_train_s.shape}")
print(f"Shape of X_test:  {X_test_s.shape}")

# --- 3.2. Train the Random Forest Classifier ---

print("\nTraining Random Forest on UNBALANCED structured data...")
# Limiting max_depth to 15 to prevent it from memorizing the massive dataset
# (and to save RAM/Time)
rf_struct = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_struct.fit(X_train_s, y_train_s)
print("Training complete.")

# --- 3.3. Evaluate ---
print("\nMaking predictions on Test Set...")
y_pred_s = rf_struct.predict(X_test_s)

print("\n--- Baseline 1: Structured Metadata (Unbalanced) ---")
print(classification_report(y_test_s, y_pred_s, target_names=['Failure', 'Success']))

# Plot Confusion Matrix
with plt.style.context('default'):
  fig, ax = plt.subplots(figsize=(6, 5))
  ConfusionMatrixDisplay.from_predictions(
      y_test_s,
      y_pred_s,
      ax=ax,
      display_labels=['Failure', 'Success'],
      cmap='Blues'
  )
  plt.title("Baseline 1: Structured (Unbalanced)")
  plt.show()

In [ ]:
# Run Baseline 2 (Embeddings Unbalanced).

def process_embeddings_for_baseline(df):
    """
    1. Filters for definitive outcomes.
    2. Stacks and concatenates the two embedding columns into a single matrix.
    Returns: X (numpy array), y (pandas series)
    """
    # 1. Filter Target
    success = ["COMPLETED"]
    failure = ["TERMINATED", "WITHDRAWN", "SUSPENDED"]
    definitive = success + failure

    df_clean = df[df['overall_status'].isin(definitive)].copy()
    y_clean = df_clean['overall_status'].apply(lambda x: 1 if x in success else 0)

    # 2. Stack Embeddings
    print(f"Processing {len(df_clean)} rows...")
    summary_emb = np.vstack(df_clean['brief_summary_embedding'].values)
    criteria_emb = np.vstack(df_clean['eligibility_criteria_embedding'].values)

    # 3. Concatenate
    X_clean = np.concatenate([summary_emb, criteria_emb], axis=1)

    return X_clean, y_clean

# --- Main Execution ---

# 1. Prepare Data
print("Preparing Training Data...")
X_train_emb, y_train_emb = process_embeddings_for_baseline(X_train)

print(f"Training Matrix Shape: {X_train_emb.shape}")

# 2. Train Model
print("Training Random Forest on UNBALANCED Embeddings...")
rf_emb = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    verbose=2
)
rf_emb.fit(X_train_emb, y_train_emb)
print("Training complete.")

# 3. Cleanup Training Data (Optional with 53GB, but good practice)
del X_train_emb, y_train_emb
gc.collect()

# 4. Prepare Test Data
print("Preparing Test Data...")
X_test_emb, y_test_emb = process_embeddings_for_baseline(X_test)

# 5. Evaluate
print("Making predictions...")
y_pred_emb = rf_emb.predict(X_test_emb)

print("\n--- Baseline 2: Embeddings (Unbalanced) ---")
print(classification_report(y_test_emb, y_pred_emb, target_names=['Failure', 'Success']))

with plt.style.context('default'):
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test_emb,
        y_pred_emb,
        ax=ax,
        display_labels=['Failure', 'Success'],
        cmap='Purples'
    )
    plt.title("Baseline Embeddings (Unbalanced)")
    plt.show()

# Final Cleanup
del X_test_emb, y_test_emb, rf_emb
gc.collect()

## Discussion:

The results from our initial baselines showcase the challenges of training on severely imbalanced data (approx. 86% Success vs. 14% Failure).

*   **Baseline 1 (Structured):** While achieving a high accuracy of **91%**, the model missed **56%** of all actual failures (Recall: 0.44).
*   **Baseline 2 (Embeddings):** The issue is even more extreme here. Despite an accuracy of **86%**, the model failed to identify a **single failed trial** (Recall: 0.00).

These models exhibit the "Lazy Classifier" phenomenon: they maximize their global accuracy by simply defaulting to the majority class ("Success"). In a clinical context, this is a critical failure mode; a model that never flags a risky trial is functionally useless, regardless of its accuracy.

**Conclusion:** The predictive signal for failure exists, but it is being overwhelmed by the majority class. To uncover this signal, we must alter the training distribution. In the next section, we implement a **Downsampling Strategy** to force the models to learn the characteristics of failed trials, accepting that this will likely increase False Positives (lower Precision) in exchange for the necessary Recall.


# Data Balancing Strategy

Explain the downsampling and what the balance_raw_data does.

In [ ]:
# Load and balance

print("Reloading X_train for balancing...")
# We need to reload because we might have deleted parts of it to save RAM
# Ensure this path matches your setup
base_path = '/content/drive/My Drive/clinical_trials_embeddings/'
X_train = pd.read_parquet(base_path + 'train_set.parquet', memory_map=True)

print("Balancing Training Data...")
# This uses your function from utils.py
X_train_balanced = balance_raw_data(X_train)

# Clean up the huge unbalanced file immediately to save RAM
del X_train
gc.collect()

print(f"Balanced Train Shape: {X_train_balanced.shape}")

#  Baselines Phase 2: Recovering the Signal

We now retrain on balanced data to prioritize recall

In [ ]:
# Run Baseline 3 (Structured Balanced).

# Preprocess Balanced Structured Data
# We create a lightweight version first to avoid passing embeddings to the preprocessor
structured_cols = [
    'phases', 'study_type', 'enrollment_count',
    'lead_sponsor_class', 'sex', 'minimum_age', 'maximum_age', 'overall_status'
]

X_train_bal_struct_raw = X_train_balanced[structured_cols].copy()

print("Preprocessing BALANCED Structured Data...")
# Learn NEW rules from the balanced set (e.g., median age might change)
processed_train_bal_struct, balanced_rules = fit_and_preprocess_train(X_train_bal_struct_raw)

# Prepare Data
X_train_sb = processed_train_bal_struct.drop('target', axis=1)
y_train_sb = processed_train_bal_struct['target']

# We reuse the processed_test_struct from Baseline 1
# (But technically we should re-transform it with balanced_rules to be 100% strict)
# Let's re-transform to be scientifically accurate:
X_test_struct_raw = X_test[structured_cols].copy() # Ensure X_test is loaded!
X_test_struct_raw['overall_status'] = X_test['overall_status']
processed_test_bal_struct = transform_test_data(X_test_struct_raw, balanced_rules)

X_test_sb = processed_test_bal_struct.drop('target', axis=1)
y_test_sb = processed_test_bal_struct['target']

# Train & Evaluate
print("Training RF on BALANCED Structured Data...")
rf_struct_bal = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_struct_bal.fit(X_train_sb, y_train_sb)

print("Evaluating...")
y_pred_sb = rf_struct_bal.predict(X_test_sb)

print("\n--- Baseline 3: Structured Metadata (Balanced) ---")
print(classification_report(y_test_sb, y_pred_sb, target_names=['Failure', 'Success']))

with plt.style.context('default'):
  fig, ax = plt.subplots(figsize=(6, 5))
  ConfusionMatrixDisplay.from_predictions(y_test_sb, y_pred_sb, ax=ax, display_labels=['Failure', 'Success'], cmap='Blues')
  plt.title("Baseline 3: Structured (Balanced)")
  plt.show()


In [ ]:
# Run Baseline 4 (Embeddings Balanced).

# Prepare Balanced Embeddings
print("Stacking BALANCED Training Embeddings...")
# X_train_balanced already has the definitive stati filtered (because balance_raw_data does it)
# So we just need to stack.

train_emb_summary = np.vstack(X_train_balanced['brief_summary_embedding'].values)
train_emb_criteria = np.vstack(X_train_balanced['eligibility_criteria_embedding'].values)
X_train_emb_bal = np.concatenate([train_emb_summary, train_emb_criteria], axis=1)

# Generate target (Map string to 0/1)
success_stati = ["COMPLETED"]
y_train_emb_bal = X_train_balanced['overall_status'].apply(lambda x: 1 if x in success_stati else 0)

# Cleanup numpy arrays
del train_emb_summary, train_emb_criteria
gc.collect()

# Train
print("Training RF on BALANCED Embeddings...")
# This is much faster now because we only have ~90k rows instead of ~380k!
rf_emb_bal = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, verbose=2)
rf_emb_bal.fit(X_train_emb_bal, y_train_emb_bal)

# Evaluate
print("Evaluating on Unbalanced Test Set...")

# Filter Test set to definitive only
def definitive_filter(df):
    success = ["COMPLETED"]
    failure = ["TERMINATED", "WITHDRAWN", "SUSPENDED"]
    return df[df['overall_status'].isin(success + failure)].copy()

X_test_clean = definitive_filter(X_test)

test_emb_summary = np.vstack(X_test_clean['brief_summary_embedding'].values)
test_emb_criteria = np.vstack(X_test_clean['eligibility_criteria_embedding'].values)
X_test_emb = np.concatenate([test_emb_summary, test_emb_criteria], axis=1)
y_test_emb = X_test_clean['overall_status'].apply(lambda x: 1 if x in ["COMPLETED"] else 0)

y_pred_emb_bal = rf_emb_bal.predict(X_test_emb)

print("\n--- Baseline 4: Embeddings (Balanced) ---")
print(classification_report(y_test_emb, y_pred_emb_bal, target_names=['Failure', 'Success']))

with plt.style.context('default'):
  fig, ax = plt.subplots(figsize=(6, 5))
  ConfusionMatrixDisplay.from_predictions(y_test_emb, y_pred_emb_bal, ax=ax, display_labels=['Failure', 'Success'], cmap='Purples')
  plt.title("Baseline 4: Embeddings (Balanced)")
  plt.show()


## Discussion

Analyze the tradeoff (Accuracy drops, Recall rises).

# The Multimodal Fusion Model

Explain the architecture (Intermediate Fusion).

In [ ]:
# Multimodal training

# Import helpers from your utils file
from utils import ClinicalTrialDataset, train_model

# --- 1. Define the Network Architecture (Essential Logic) ---
class MultimodalNet(nn.Module):
    def __init__(self, structured_input_dim, embedding_input_dim=1536):
        super(MultimodalNet, self).__init__()

        # Branch 1: Structured Data
        self.struct_layer1 = nn.Linear(structured_input_dim, 64)
        self.struct_bn1 = nn.BatchNorm1d(64)
        self.struct_dropout = nn.Dropout(0.3)

        # Branch 2: Text Embeddings
        self.text_layer1 = nn.Linear(embedding_input_dim, 512)
        self.text_bn1 = nn.BatchNorm1d(512)
        self.text_dropout1 = nn.Dropout(0.5)

        self.text_layer2 = nn.Linear(512, 128)
        self.text_bn2 = nn.BatchNorm1d(128)

        # Fusion
        fusion_dim = 64 + 128

        # Head
        self.fc_final = nn.Linear(fusion_dim, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x_structured, x_text):
        # Structured Path
        out_s = self.struct_layer1(x_structured)
        out_s = self.struct_bn1(out_s)
        out_s = F.relu(out_s)
        out_s = self.struct_dropout(out_s)

        # Text Path
        out_t = self.text_layer1(x_text)
        out_t = self.text_bn1(out_t)
        out_t = F.relu(out_t)
        out_t = self.text_dropout1(out_t)

        out_t = self.text_layer2(out_t)
        out_t = self.text_bn2(out_t)
        out_t = F.relu(out_t)

        # Concatenate
        combined = torch.cat((out_s, out_t), dim=1)

        # Final Prediction
        out = self.fc_final(combined)
        out = F.relu(out)
        logits = self.output(out)
        return logits

# --- 2. Prepare Data for PyTorch ---
print("Preparing Tensors for Fusion Model...")

# Helper to stack embeddings and align structured data
def prepare_tensors(processed_df, raw_df_with_embeddings):
    # 1. Get Structured Features (Drop target)
    X_struct = processed_df.drop('target', axis=1).values.astype(np.float32)

    # 2. Get Targets
    y = processed_df['target'].values.astype(np.float32)

    # 3. Get Embeddings (Align by Index)
    # We filter the raw df to match the processed df's index
    aligned_raw = raw_df_with_embeddings.loc[processed_df.index]

    emb_summary = np.vstack(aligned_raw['brief_summary_embedding'].values)
    emb_criteria = np.vstack(aligned_raw['eligibility_criteria_embedding'].values)
    X_emb = np.concatenate([emb_summary, emb_criteria], axis=1).astype(np.float32)

    return X_struct, X_emb, y

# A. Prepare TRAIN (Balanced)
# processed_train_bal_struct came from Baseline 3
# X_train_balanced came from Step 1 of Phase 2
train_struct, train_emb, train_labels = prepare_tensors(processed_train_bal_struct, X_train_balanced)

# B. Prepare VAL (Unbalanced) - For monitoring
# processed_val_struct came from Baseline 1 (Step 3)
# X_val came from Step 2.2
val_struct, val_emb, val_labels = prepare_tensors(processed_val_struct, X_val)

print(f"Train Tensor Shapes: Struct {train_struct.shape}, Emb {train_emb.shape}")
print(f"Val Tensor Shapes:   Struct {val_struct.shape},   Emb {val_emb.shape}")

# --- 3. Create Datasets & Loaders ---
train_dataset = ClinicalTrialDataset(train_struct, train_emb, train_labels)
val_dataset = ClinicalTrialDataset(val_struct, val_emb, val_labels)

# Batch size 64 or 128 is good for GPU
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

# --- 4. Initialize Model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

model = MultimodalNet(
    structured_input_dim=train_struct.shape[1], # Should be ~23-25
    embedding_input_dim=1536
)

# --- 5. Train ---
print("Starting Fusion Model Training...")
# We use the train_model function you put in utils.py
# 10 Epochs should be enough to see convergence
history = train_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.001)

print("Training Complete.")

In [ ]:
# --- 6. Prepare TEST Data (Unbalanced) ---
# processed_test_struct came from Baseline 1
# X_test came from Step 2.2
print("Preparing Test Set...")
test_struct, test_emb, test_labels = prepare_tensors(processed_test_struct, X_test)
test_dataset = ClinicalTrialDataset(test_struct, test_emb, test_labels)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# --- 7. Evaluate ---
model.eval()
all_preds = []
all_labels = []

print("Running Inference on Test Set...")
with torch.no_grad():
    for batch_struct, batch_emb, batch_labels in test_loader:
        # Move to GPU
        batch_struct = batch_struct.to(device)
        batch_emb = batch_emb.to(device)

        # Predict
        logits = model(batch_struct, batch_emb)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())

print("\n--- Model 5: Multimodal Fusion (Balanced Train) ---")
print(classification_report(all_labels, all_preds, target_names=['Failure', 'Success']))

with plt.style.context('default'):
  fig, ax = plt.subplots(figsize=(6, 5))
  ConfusionMatrixDisplay.from_predictions(all_labels, all_preds, ax=ax, display_labels=['Failure', 'Success'], cmap='Greens')
  plt.title("Model 5: Multimodal Fusion")
  plt.show()


## Discussion of the performance so far

# Post-Hoc Analysis: Threshold Optimization

In [ ]:
# Run the loop checking thresholds 0.1 to 0.5.
def optimize_threshold(model, loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    all_probs = []
    all_labels = []

    # 1. Get raw probabilities
    with torch.no_grad():
        for batch_struct, batch_emb, batch_labels in loader:
            batch_struct = batch_struct.to(device)
            batch_emb = batch_emb.to(device)

            logits = model(batch_struct, batch_emb)
            probs = torch.sigmoid(logits) # Convert logit to 0.0-1.0

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch_labels.cpu().numpy())

    all_probs = np.array(all_probs).flatten()
    all_labels = np.array(all_labels).flatten()

    # 2. Test thresholds from 0.50 to 0.95
    print(f"{'Threshold':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
    print("-" * 46)

    best_f1 = 0
    best_thresh = 0

    for thresh in np.arange(0.10, 0.51, 0.05):
        # Apply threshold
        preds = (all_probs > thresh).astype(int)

        pred_labels = (all_probs > thresh).astype(int)

        p = precision_score(all_labels, pred_labels, pos_label=0, zero_division=0)
        r = recall_score(all_labels, pred_labels, pos_label=0, zero_division=0)
        f1 = f1_score(all_labels, pred_labels, pos_label=0, zero_division=0)

        print(f"{thresh:.2f}       | {p:.4f}     | {r:.4f}     | {f1:.4f}")

# Run it on the Test Loader
optimize_threshold(model, test_loader)

## Discussion:

Explain that while 0.50 gives the best Recall, the model can be tuned for Precision if needed.

# Conclusion & Artifacts

In [ ]:
# Summary Table: Compare all 5 models side-by-side.

## Final Conclusion: Summary of findings.

In [ ]:
# Code to save the .pth file to Drive.

# 1. Define where to save (Your Google Drive)
# Ensure this path matches where you want it
save_path = '/content/drive/My Drive/clinical_trials_embeddings/fusion_model_final.pth'

# 2. Create the "Checkpoint" Dictionary
# This saves the brain (weights) AND the instructions (preprocessing rules)
checkpoint = {
    'model_state_dict': model.state_dict(),
    'preprocessing_rules': balanced_rules, # From Baseline 3/Fusion setup
    'hyperparameters': {
        'structured_dim': train_struct.shape[1],
        'embedding_dim': 1536,
        'threshold': 0.50
    }
}

# 3. Save it
print(f"Saving model artifact to {save_path}...")
torch.save(checkpoint, save_path)

print("Model saved successfully!")
print(f"File size: {os.path.getsize(save_path) / 1024 / 1024:.2f} MB")

In [ ]:
# Code to prove the saved file works.

# --- Demo: How to Load the Saved Model ---
print("Testing Model Loading...")

# 1. Load the file
checkpoint = torch.load('/content/drive/My Drive/clinical_trials_embeddings/fusion_model_final.pth')

# 2. Restore Preprocessing Rules
loaded_rules = checkpoint['preprocessing_rules']
input_dim = checkpoint['hyperparameters']['structured_dim']

# 3. Initialize Model Architecture
# (We need to re-instantiate the class structure first)
loaded_model = MultimodalNet(structured_input_dim=input_dim, embedding_input_dim=1536)

# 4. Load Weights
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.to(device)
loaded_model.eval()

print("Model loaded and ready for inference.")
print(f"Restored Scaler Mean: {loaded_rules['scaler'].mean_[:5]}") # Proof that rules are back